# RGB Extraction v4 — Fitur A/B

Memuat **CNN RGB v4 yang sudah dilatih**, mengekstrak 512-D `svm_features` pada seluruh 50.000 gambar official training, untuk original (A) dan original+flip (B). AVG/NTSC tidak diekstraksi ulang. Official test tidak dibuka. Notebook ini **tidak melatih CNN maupun SVM**. Hasil dipakai dalam Feature Engineering Combination v4.

## 1. Konfigurasi dan metadata CNN RGB v4

Notebook menggunakan kernel `envGPU` dan folder `tugas-7`. Pastikan `RGB Training v4` selesai. Identitas model diambil dari SHA-256 sehingga checkpoint yang tertukar langsung ditolak.

In [1]:
from pathlib import Path
import gc,hashlib,json,os,time
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
SEED=30092026
RUN_ID='cifar10_rgb_residual_v4'
ROOT=Path.cwd().resolve()
if not (ROOT/'outputs'/RUN_ID).is_dir() and (ROOT/'tugas-7'/'outputs'/RUN_ID).is_dir():ROOT=ROOT/'tugas-7'
RGB_OUT=ROOT/'outputs'/RUN_ID/'rgb'
MODEL_PATH=RGB_OUT/'model'/'cnn.keras'
REPORT_PATH=RGB_OUT/'laporan'/'cnn_training.json'
FEATURE_PATH=RGB_OUT/'fitur'/'train_features.npz'
BATCH_SIZE=128
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu,True)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for piece in iter(lambda:f.read(1024*1024),b''):h.update(piece)
    return h.hexdigest()
if not MODEL_PATH.is_file() or not REPORT_PATH.is_file():
    raise FileNotFoundError('Jalankan RGB Training v4 terlebih dahulu')
metadata=json.loads(REPORT_PATH.read_text(encoding='utf-8'))
assert metadata['seed']==SEED and metadata['run_id']==RUN_ID
assert sha256_file(MODEL_PATH)==metadata['cnn_sha256']
print('Model terverifikasi:',metadata['cnn_sha256'][:16])

I0000 00:00:1791510880.245776  253669 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791510880.282568  253669 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791510881.208776  253669 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Model terverifikasi: cea218a9171acf35


## 2. Split dan label official training

Data official test tidak dimuat. `train_idx`, `val_idx`, dan `split_hash` harus identik dengan training RGB serta arsip AVG/NTSC sebelumnya.

In [2]:
from tensorflow.keras.datasets import cifar10
(x_all,y_all),_=cifar10.load_data()
y_all=y_all.ravel().astype(np.int64)
train_idx,val_idx=train_test_split(np.arange(50000),test_size=.2,stratify=y_all,random_state=SEED)
train_idx,val_idx=np.sort(train_idx),np.sort(val_idx)
split_hash=hashlib.sha256(train_idx.tobytes()+val_idx.tobytes()).hexdigest()
if split_hash!=metadata['split_hash']:raise ValueError('Split berbeda dari RGB Training v4')
print('Split sesuai:',split_hash[:16])

Split sesuai: 16004a1ac8a89ac9


## 3. Extractor pooling 512-D beku

Ambil layer bernama `svm_features`. Keluaran `training=False` menonaktifkan dropout dan augmentasi acak. Variabel `features_A` dan `features_B` dihitung dari representasi model yang sama secara konsisten.

In [3]:
cnn=tf.keras.models.load_model(MODEL_PATH,compile=False)
extractor=tf.keras.Model(cnn.input,cnn.get_layer('svm_features').output)
if int(extractor.output.shape[-1])!=512:raise ValueError('Fitur RGB tidak berukuran 512')
features_A=np.empty((50000,512),dtype=np.float32)
features_B=np.empty((50000,512),dtype=np.float32)
print('Layer:',cnn.get_layer('svm_features').name,'| Dimensi:',extractor.output.shape)

I0000 00:00:1791510883.927922  253669 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2872 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Layer: svm_features | Dimensi: (None, 512)


## 4. Ekstraksi original dan horizontal flip

Proses setiap batch untuk membatasi VRAM. A adalah output original; B adalah rata-rata original dan flip horizontal. Rata-rata dilakukan sebelum penskalaan feature-engineering.

In [4]:
start=time.perf_counter()
for i,begin in enumerate(range(0,len(x_all),BATCH_SIZE),1):
    end=min(begin+BATCH_SIZE,len(x_all))
    batch=x_all[begin:end].astype(np.float32)/255.0
    original=extractor(batch,training=False).numpy().astype(np.float32)
    horizontal=extractor(np.ascontiguousarray(batch[:,:,::-1,:]),training=False).numpy().astype(np.float32)
    features_A[begin:end]=original
    features_B[begin:end]=(original+horizontal)*np.float32(0.5)
    if i%20==0 or end==len(x_all):print(f'RGB v4: {end}/{len(x_all)}',flush=True)
assert np.isfinite(features_A).all() and np.isfinite(features_B).all()
print('Durasi detik:',round(time.perf_counter()-start,1))
del extractor,cnn;tf.keras.backend.clear_session();gc.collect()

E0000 00:00:1791510885.174967  253669 util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
I0000 00:00:1791510885.202608  253669 cuda_dnn.cc:461] Loaded cuDNN version 92600


RGB v4: 2560/50000
RGB v4: 5120/50000
RGB v4: 7680/50000
RGB v4: 10240/50000
RGB v4: 12800/50000
RGB v4: 15360/50000
RGB v4: 17920/50000
RGB v4: 20480/50000
RGB v4: 23040/50000
RGB v4: 25600/50000
RGB v4: 28160/50000
RGB v4: 30720/50000
RGB v4: 33280/50000
RGB v4: 35840/50000
RGB v4: 38400/50000
RGB v4: 40960/50000
RGB v4: 43520/50000
RGB v4: 46080/50000
RGB v4: 48640/50000
RGB v4: 50000/50000
Durasi detik: 50.9


0

## 5. Simpan fitur dan provenance

Arsip memuat kedua representasi, label, indeks asli, split, run ID dan hash checkpoint. Jika file sudah ada, jalankan pemeriksaan identitas dan **jangan menimpanya**.

In [5]:
if FEATURE_PATH.exists():
    with np.load(FEATURE_PATH,allow_pickle=False) as old:
        if (str(old['cnn_sha256'])!=metadata['cnn_sha256'] or str(old['split_hash'])!=split_hash
            or old['features_A'].shape!=(50000,512) or old['features_B'].shape!=(50000,512)):
            raise ValueError('Arsip RGB v4 lama tidak kompatibel')
    print('Arsip sebelumnya sudah ada. Tidak menimpanya:',FEATURE_PATH)
else:
    tmp=FEATURE_PATH.with_suffix('.npz.tmp')
    with open(tmp,'wb') as stream:
        np.savez_compressed(stream,features_A=features_A,features_B=features_B,
            y=y_all,indices=np.arange(50000),train_idx=train_idx,val_idx=val_idx,
            run_id=RUN_ID,split_hash=split_hash,cnn_sha256=metadata['cnn_sha256'],
            layer_name='svm_features',feature_dim=512,preprocessing_version='rgb-v4-inference-float32-div255')
    os.replace(tmp,FEATURE_PATH)
    print('Berhasil disimpan:',FEATURE_PATH,'| SHA:',sha256_file(FEATURE_PATH)[:16])

Arsip sebelumnya sudah ada. Tidak menimpanya: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_rgb_residual_v4/rgb/fitur/train_features.npz
